# Number Systems for AI/ML

Every computation in machine learning — from gradient updates to inference — depends on how numbers are stored and manipulated. This notebook covers number systems **from first principles to real PyTorch implementations**.

| Section | Topic | Real-World Skill |
|---------|-------|------------------|
| 1 | Number Types & Memory | Choosing dtypes for tensors |
| 2 | IEEE 754 Float Internals | Debugging NaN and precision bugs |
| 3 | Overflow & Underflow | Stable softmax, log-sum-exp |
| 4 | Complex Numbers & FFT | Signal processing, positional encoding |
| 5 | Binary & Bitwise Ops | Memory layout, efficient tricks |
| 6 | Mixed Precision Training | Real `torch.cuda.amp` usage |
| 7 | Quantization | PTQ, per-channel, GPTQ/AWQ concepts |
| 8 | Modern GPU Formats | FP8, TF32, BFloat16 |
| 9 | Numerical Algorithms | Kahan summation, stochastic rounding |

**Prerequisites**: Basic Python, NumPy fundamentals

In [ ]:
import numpy as np
import struct
import warnings
np.random.seed(42)
np.set_printoptions(precision=6, suppress=True)

# Optional: PyTorch for real implementations
try:
    import torch
    import torch.nn as nn
    torch.manual_seed(42)
    HAS_TORCH = True
    print(f'NumPy {np.__version__} | PyTorch {torch.__version__}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name()}')
except ImportError:
    HAS_TORCH = False
    print(f'NumPy {np.__version__} | PyTorch not installed (NumPy demos still work)')

# Optional: SciPy for reference implementations
try:
    from scipy.special import logsumexp as scipy_logsumexp
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

---

## 1. Number Types & Memory Management

### The Number Hierarchy

```
ℂ (Complex) ⊃ ℝ (Real) ⊃ ℚ (Rational) ⊃ ℤ (Integer) ⊃ ℕ (Natural)
```

| Symbol | Set | NumPy Type | ML Use Case |
|--------|-----|------------|-------------|
| ℕ | Natural (0,1,2..) | `uint8/16` | Pixel values, indices |
| ℤ | Integers | `int8/32/64` | Token IDs, quantized weights |
| ℝ | Reals | `float16/32/64` | Weights, gradients, losses |
| ℂ | Complex | `complex64/128` | FFT, eigenvalues |

In [ ]:
# Python's arbitrary-precision integers vs NumPy's fixed-width integers
big_int = 10**100  # Python handles this natively
print(f'Python integer 10^100: {len(str(big_int))} digits (no overflow!)')

# But NumPy integers have fixed widths → overflow risk!
print(f'np.int8(127) + 1 = {np.int8(np.int8(127).astype(np.int16) + 1)}  ← wraps to -128!')

# Memory comparison table — critical for large model training
print('\n' + '='*65)
print(f'{"Type":<12} {"Bytes":<6} {"Min":>20} {"Max":>20}')
print('-'*65)
for dt in [np.int8, np.int16, np.int32, np.int64]:
    i = np.iinfo(dt)
    print(f'{dt.__name__:<12} {dt().nbytes:<6} {i.min:>20,} {i.max:>20,}')

print(f'\n{"Type":<12} {"Bytes":<6} {"Precision":<14} {"ML Role"}')
print('-'*65)
for dt, prec, role in [
    (np.float16, '~3 digits', 'Inference, mixed precision forward'),
    (np.float32, '~7 digits', 'Standard training (default)'),
    (np.float64, '~15 digits', 'Loss accumulation, scientific'),
]:
    print(f'{dt.__name__:<12} {dt().nbytes:<6} {prec:<14} {role}')

# Real memory impact
n_params = 7_000_000_000  # 7B parameter model
print(f'\n--- Memory for {n_params/1e9:.0f}B parameter model ---')
for dt in [np.float32, np.float16, np.int8, np.int4 if hasattr(np, 'int4') else None]:
    if dt is None:
        print(f'  int4:    {n_params * 0.5 / 1e9:>6.1f} GB  ← GGUF/GPTQ 4-bit')
    else:
        print(f'  {dt.__name__:<8}  {n_params * dt().nbytes / 1e9:>6.1f} GB')

---

## 2. IEEE 754: How Floats Actually Work

A float stores: `(-1)^sign × 1.mantissa × 2^(exponent - bias)`

```
64-bit float (float64):
┌───┬──────────────┬────────────────────────────────────────────────────────┐
│ S │  Exponent    │                    Mantissa (52 bits)                  │
│1b │   11 bits    │                                                        │
└───┴──────────────┴────────────────────────────────────────────────────────┘

32-bit float (float32):
┌───┬──────────┬───────────────────────────────┐
│ S │ Exponent │       Mantissa (23 bits)       │
│1b │  8 bits  │                                 │
└───┴──────────┴───────────────────────────────┘

16-bit float (float16):
┌───┬──────┬────────────────┐
│ S │ Exp  │ Mantissa (10b) │
│1b │ 5b   │                 │
└───┴──────┴────────────────┘
```

In [ ]:
# Inspect the ACTUAL binary representation using struct

def float_to_binary(value, fmt='f'):
    """Show exact binary layout of a float.
    fmt: 'f' for float32, 'd' for float64
    """
    packed = struct.pack(f'>{fmt}', value)
    bits = ''.join(f'{byte:08b}' for byte in packed)
    if fmt == 'f':  # 32-bit
        return f'{bits[0]} | {bits[1:9]} | {bits[9:]}'
    else:  # 64-bit
        return f'{bits[0]} | {bits[1:12]} | {bits[12:]}'

print('IEEE 754 Binary Representations:')
print('='*55)
for val in [1.0, -1.0, 0.1, 0.5, 3.14, float('inf'), float('nan')]:
    bits = float_to_binary(val)
    print(f'  {val:>8}  →  S|Exponent |Mantissa')
    print(f'  {"":>8}     {bits}')

# Key insight: 0.1 has an infinite binary expansion!
print('\n--- Why 0.1 + 0.2 ≠ 0.3 ---')
print(f'  0.1 stored as: {0.1:.20f}')
print(f'  0.2 stored as: {0.2:.20f}')
print(f'  0.1 + 0.2    = {0.1+0.2:.20f}')
print(f'  0.3 stored as: {0.3:.20f}')
print(f'  Equal? {0.1 + 0.2 == 0.3}  →  Use np.isclose(): {np.isclose(0.1+0.2, 0.3)}')

In [ ]:
# Machine Epsilon — the smallest difference a float can detect
# CRITICAL: learning rates smaller than ε have NO EFFECT!

print('Machine Epsilon (ε) — minimum meaningful difference from 1.0:')
print('-'*60)
for dtype in [np.float16, np.float32, np.float64]:
    eps = np.finfo(dtype).eps
    one = dtype(1.0)
    test1 = (one + dtype(eps)) != one      # True: ε is detectable
    test2 = (one + dtype(eps / 2)) == one   # True: ε/2 is lost!
    print(f'  {dtype.__name__:<10} ε = {eps:<12.2e}  1+ε≠1: {test1}  1+ε/2=1: {test2}')

print(f'\n⚠ For float32: lr < {np.finfo(np.float32).eps:.0e} is effectively zero!')
print('⚠ For float16: lr < 9.77e-04 is effectively zero!')
print('→ This is why mixed precision uses float32 for weight updates')

# Demonstrate precision loss in gradient accumulation
print('\n--- Precision loss in gradient accumulation ---')
large = np.float32(1e7)
small = np.float32(1.0)
print(f'  float32: {large} + {small} = {np.float32(large + small)}')
print(f'  Expected: {large + small:.1f}')
print(f'  Lost! The small value vanishes when added to a large accumulator')

---

## 3. Overflow & Underflow — The Two Killers

| Issue | Description | Symptom | Fix |
|-------|-------------|---------|-----|
| **Overflow** | Number too large → `inf` | `loss = NaN` | Subtract max (softmax), gradient clipping |
| **Underflow** | Number too small → `0` | Probabilities vanish | Work in log-space |

### Softmax: The Classic Overflow Problem

$$\sigma(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

When $x_i$ is large, $e^{x_i} \to \infty$. **Fix**: subtract max first (mathematically equivalent).

In [ ]:
# Softmax: Naive vs Stable

def naive_softmax(x):
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x)

def stable_softmax(x):
    """Used in ALL production frameworks."""
    exp_x = np.exp(x - np.max(x))  # subtract max
    return exp_x / np.sum(exp_x)

# Normal case — both work
x_normal = np.array([1.0, 2.0, 3.0])
print('Normal logits [1, 2, 3]:')
print(f'  Naive:  {naive_softmax(x_normal)}')
print(f'  Stable: {stable_softmax(x_normal)}')

# Large logits — naive breaks!
x_large = np.array([1000.0, 1001.0, 1002.0])
print(f'\nLarge logits [1000, 1001, 1002]:')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print(f'  Naive:  {naive_softmax(x_large)}  ← NaN!')
print(f'  Stable: {stable_softmax(x_large)}  ← Works!')

# Real PyTorch comparison
if HAS_TORCH:
    logits = torch.tensor([1000.0, 1001.0, 1002.0])
    print(f'\n  PyTorch F.softmax: {torch.nn.functional.softmax(logits, dim=0).numpy()}')
    print('  → PyTorch uses the stable version internally')

In [ ]:
# Underflow: Product of Small Probabilities
# P(sentence) = P(w₁)×P(w₂|w₁)×P(w₃|w₁,w₂)×...

small_probs = np.array([0.001] * 100)  # 100 words, P=0.001 each

direct_product = np.prod(small_probs)
print(f'Product of 100 × 0.001: {direct_product}  ← Underflow to 0!')

# Fix: work in log-space
log_prob = np.sum(np.log(small_probs))
print(f'Sum of log probs: {log_prob:.2f}  ← Tracked without underflow!')
print(f'→ To compare: higher log-prob = more likely')

# Log-Sum-Exp Trick — essential for cross-entropy loss
print('\n--- Log-Sum-Exp Trick ---')

def logsumexp_naive(x):  return np.log(np.sum(np.exp(x)))
def logsumexp_stable(x):
    m = np.max(x)
    return m + np.log(np.sum(np.exp(x - m)))

x = np.array([1000, 1001, 1002])
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print(f'  Naive:  {logsumexp_naive(x)}  ← inf!')
print(f'  Stable: {logsumexp_stable(x):.4f}')

if HAS_SCIPY:
    print(f'  SciPy:  {scipy_logsumexp(x):.4f}')

if HAS_TORCH:
    print(f'  PyTorch: {torch.logsumexp(torch.tensor(x, dtype=torch.float64), dim=0).item():.4f}')

---

## 4. Complex Numbers in ML

Complex: $z = a + bi$ where $i^2 = -1$

| Application | Usage |
|-------------|-------|
| **FFT** | Audio/image processing, efficient convolutions |
| **Eigenvalues** | PCA, spectral clustering |
| **Rotary Embeddings** | RoPE in LLaMA, GPT-NeoX (position encoding) |
| **Quantum ML** | Quantum circuit simulation |

In [ ]:
# Complex Numbers — Operations and ML Applications

z1 = 3 + 4j
print(f'z = {z1}  |  Real: {z1.real}  Imag: {z1.imag}')
print(f'|z| = √(3²+4²) = {abs(z1)}')
print(f'z × z* = |z|² = {z1 * z1.conjugate()}  (always real!)')

# Euler's formula: e^(iπ) = -1
euler = np.exp(1j * np.pi)
print(f'\ne^(iπ) = {euler.real:.0f} + {euler.imag:.2e}i ≈ -1  (Euler\'s identity)')

# FFT — frequency decomposition
print('\n--- FFT: Detecting Frequencies ---')
t = np.linspace(0, 1, 256)
signal = np.sin(2*np.pi*5*t) + 0.5*np.sin(2*np.pi*20*t)  # 5Hz + 20Hz
fft = np.fft.fft(signal)
freqs = np.fft.fftfreq(len(signal), t[1]-t[0])
peaks = np.argsort(np.abs(fft))[-4:-1][::-1]
print(f'  Signal: sin(2π·5t) + 0.5·sin(2π·20t)')
print(f'  FFT dtype: {fft.dtype}  (complex!)')
print(f'  Detected frequencies: {freqs[peaks[:2]].astype(int)} Hz  ✓')

# Rotary Position Embeddings (RoPE) — used in LLaMA, GPT-NeoX
print('\n--- RoPE: Complex Rotation for Position Encoding ---')
d = 4  # embedding dimension
pos = 3  # position in sequence
theta = 10000 ** (-2 * np.arange(d//2) / d)  # frequency bases
rotation = np.exp(1j * pos * theta)  # complex rotation
print(f'  Position {pos}, dim {d}')
print(f'  θ = {theta.round(4)}')
print(f'  Rotation = e^(i·{pos}·θ) = {rotation.round(4)}')
print('  → Multiply query/key embeddings by rotation → position-aware attention')
print('  → This is how LLaMA encodes token positions!')

---

## 5. Binary Representation & Bitwise Operations

| Base | Name | Python | ML Use |
|------|------|--------|--------|
| 2 | Binary | `0b1010` | Memory layout, masks |
| 16 | Hex | `0xFF` | Color codes, memory addresses |

### Two's Complement: Why int8 = [-128, 127]
- **Positive**: normal binary (42 = `00101010`)
- **Negative**: flip bits + add 1 (-42 = `11010110`)
- **8 bits**: 256 values = 128 negative + 1 zero + 127 positive

In [ ]:
# Binary breakdown and Two's Complement
n = 42
print(f'{n} in different bases:')
print(f'  Binary: {bin(n)} = {n:08b}')
print(f'  Hex:    {hex(n)}')

print(f'\nBinary breakdown of {n}:')
for i in range(7, -1, -1):
    bit = (n >> i) & 1
    if bit: print(f'  2^{i} × {bit} = {2**i:3d}')

# Two's complement — critical for quantization
print('\n--- Two\'s Complement (int8 ranges) ---')
for dt in [np.int8, np.uint8, np.int16]:
    i = np.iinfo(dt)
    print(f'  {dt.__name__:<8} {i.bits}b: [{i.min:>6}, {i.max:>6}]')

print(f'\n  Overflow: int8(127) + 1 = {np.int8(np.int8(127).astype(np.int16) + 1)}  ← wraps!')
print('  → Quantization must use np.clip() to prevent this')

# Bitwise operations used in ML
print('\n--- Bitwise Ops ---')
a, b = 0b1100, 0b1010
print(f'  {a:04b} & {b:04b} = {a&b:04b}  (AND: feature masking)')
print(f'  {a:04b} | {b:04b} = {a|b:04b}  (OR: combine flags)')
print(f'  {a:04b} ^ {b:04b} = {a^b:04b}  (XOR: hash functions)')

# Power-of-2 batch size check
is_pow2 = lambda n: n > 0 and (n & (n-1)) == 0
print('\n  Batch size validation (power of 2 for GPU efficiency):')
for bs in [16, 32, 48, 64, 128]:
    print(f'    {bs}: {"✓" if is_pow2(bs) else "✗"}')

---

## 6. Mixed Precision Training — Real PyTorch

Modern training uses **mixed precision**: float16/bfloat16 for speed, float32 for accuracy.

```
Float32:  1 sign + 8 exp + 23 mantissa  (baseline)
Float16:  1 sign + 5 exp + 10 mantissa  (more precision, less range)
BFloat16: 1 sign + 8 exp + 7 mantissa   (same range as float32!)
```

| Aspect | Float32 | Mixed Precision |
|--------|---------|----------------|
| Memory | Baseline | ~50% reduction |
| Speed | Baseline | 2-3× faster |
| Accuracy | 100% | ~Same |

In [ ]:
# Float16 vs Float32: Where precision breaks

print('Precision comparison — small gradients:')
for val in [1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8]:
    f32 = np.float32(val)
    f16 = np.float16(val)
    lost = '← LOST!' if f16 == 0 else ''
    print(f'  {val:.0e}  float32={f32:.2e}  float16={f16}  {lost}')

# Loss scaling solution
print('\n--- Loss Scaling ---')
tiny = np.float32(1e-6)
scale = 1024
print(f'  Gradient: {tiny} → float16: {np.float16(tiny)}  ← lost!')
print(f'  Scaled ×{scale}: {tiny*scale} → float16: {np.float16(tiny*scale)}  ← preserved!')
print(f'  After unscale: {np.float16(tiny*scale)/scale}  ✓')
print('  → This is what torch.cuda.amp.GradScaler does')

In [ ]:
# ══ REAL PyTorch Mixed Precision Training ══

if HAS_TORCH:
    # Simple model
    model = nn.Sequential(
        nn.Linear(784, 256),
        nn.ReLU(),
        nn.Linear(256, 10)
    )

    # Memory comparison
    fp32_bytes = sum(p.numel() * 4 for p in model.parameters())
    fp16_bytes = sum(p.numel() * 2 for p in model.parameters())
    print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
    print(f'  float32 memory: {fp32_bytes:,} bytes')
    print(f'  float16 memory: {fp16_bytes:,} bytes ({fp32_bytes/fp16_bytes:.0f}× smaller)')

    # Real AMP training loop structure
    print('\n--- Real AMP Training Code ---')
    print('''
    # How it's done in production:
    scaler = torch.cuda.amp.GradScaler()
    
    for batch in dataloader:
        optimizer.zero_grad()
        
        # Forward pass in float16 (fast!)
        with torch.cuda.amp.autocast():
            output = model(batch)
            loss = criterion(output, target)
        
        # Backward pass with scaled gradients
        scaler.scale(loss).backward()
        scaler.step(optimizer)  # unscale + update
        scaler.update()         # adjust scale factor
    ''')

    # Demonstrate dtype casting behavior
    print('--- PyTorch dtype behavior ---')
    x = torch.randn(3, 3)
    print(f'  Default: {x.dtype}')
    print(f'  .half():  {x.half().dtype}')
    print(f'  .bfloat16(): {x.bfloat16().dtype}')
    print(f'  .double(): {x.double().dtype}')

    # BFloat16 vs Float16 range test
    big = torch.tensor(65504.0)  # max float16
    print(f'\n  float16 max:   {big.half()}')
    print(f'  float16(70000): {torch.tensor(70000.0).half()}  ← overflow to inf!')
    print(f'  bfloat16(70000): {torch.tensor(70000.0).bfloat16()}  ← safe (same range as fp32)')
    print('  → BFloat16 preferred for training — wider range, fewer NaN issues')
else:
    print('PyTorch not available. Install with: pip install torch')

---

## 7. Quantization — Model Compression for Deployment

**Quantization** = converting float32 → lower precision (int8, int4)

| Level | Scale Factors | Accuracy | Used By |
|-------|--------------|----------|--------|
| Per-tensor | 1 for all | ~5-10% loss | Basic |
| Per-channel | 1 per output channel | ~1-2% loss | PyTorch, TensorRT |
| Per-group | 1 per 32-128 weights | <1% loss | GPTQ, AWQ, GGUF |

In [ ]:
# Quantization: Float32 → Int8

def quantize_int8(weights, symmetric=True):
    scale = np.max(np.abs(weights)) / 127
    q = np.clip(np.round(weights / scale), -128, 127).astype(np.int8)
    return q, scale

def dequantize_int8(q, scale):
    return q.astype(np.float32) * scale

np.random.seed(42)
weights = np.random.randn(1000).astype(np.float32)
q, scale = quantize_int8(weights)
recon = dequantize_int8(q, scale)

print(f'Original:  {weights.dtype}, {weights.nbytes:,} bytes')
print(f'Quantized: {q.dtype}, {q.nbytes:,} bytes  ({weights.nbytes/q.nbytes:.0f}× smaller)')
print(f'MSE: {np.mean((weights - recon)**2):.6f}')
print(f'Max error: {np.max(np.abs(weights - recon)):.4f}')

In [ ]:
# Per-Channel vs Per-Tensor — why it matters

def quantize_per_tensor(W):
    scale = np.max(np.abs(W)) / 127
    return np.clip(np.round(W / scale), -128, 127).astype(np.int8), scale

def quantize_per_channel(W):
    scales = np.array([np.max(np.abs(W[c])) / 127 for c in range(W.shape[0])])
    scales = np.where(scales > 0, scales, 1.0)
    q = np.zeros_like(W, dtype=np.int8)
    for c in range(W.shape[0]):
        q[c] = np.clip(np.round(W[c] / scales[c]), -128, 127).astype(np.int8)
    return q, scales

# Real scenario: channels with VERY different magnitudes
np.random.seed(42)
W = np.random.randn(8, 256).astype(np.float32)
W[0] *= 10     # Channel 0: large weights
W[1] *= 0.01   # Channel 1: tiny weights

q_t, s_t = quantize_per_tensor(W)
q_c, s_c = quantize_per_channel(W)

recon_t = q_t.astype(np.float32) * s_t
recon_c = q_c.astype(np.float32) * s_c[:, None]

print(f'{"Ch":>3} {"Weight σ":>9} {"Per-Tensor MSE":>15} {"Per-Channel MSE":>16}')
print('-'*46)
for c in range(8):
    et = np.mean((W[c]-recon_t[c])**2)
    ec = np.mean((W[c]-recon_c[c])**2)
    marker = ' ← huge error!' if et > 10*ec else ''
    print(f'  {c}   {np.std(W[c]):>8.4f}  {et:>14.6f}  {ec:>15.6f}{marker}')

total_t = np.mean((W-recon_t)**2)
total_c = np.mean((W-recon_c)**2)
print(f'\nTotal — Per-tensor: {total_t:.6f}  Per-channel: {total_c:.6f}')
print(f'Per-channel is {total_t/total_c:.1f}× more accurate!')
print('→ Ch1 (tiny weights) gets crushed by per-tensor\'s global scale')

In [ ]:
# ══ REAL PyTorch Quantization ══

if HAS_TORCH:
    # Dynamic quantization — simplest, inference only
    model = nn.Sequential(
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 10)
    )

    # Size before quantization
    orig_size = sum(p.numel() * p.element_size() for p in model.parameters())

    # Apply dynamic quantization (real PyTorch API)
    quantized_model = torch.quantization.quantize_dynamic(
        model,
        {nn.Linear},   # which layers to quantize
        dtype=torch.qint8
    )

    print('PyTorch Dynamic Quantization:')
    print(f'  Original model size: {orig_size:,} bytes')
    print(f'  Quantized layers: {nn.Linear.__name__} → qint8')

    # Verify it works
    x = torch.randn(1, 512)
    with torch.no_grad():
        y_orig = model(x)
        y_quant = quantized_model(x)

    diff = (y_orig - y_quant).abs().mean().item()
    print(f'  Output difference: {diff:.6f} (should be very small)')
    print(f'\n  Quantized model structure:')
    for name, module in quantized_model.named_modules():
        if name: print(f'    {name}: {module.__class__.__name__}')

    # Show what production LLM quantization looks like
    print('\n--- LLM Quantization Methods ---')
    print('''
    # GPTQ (4-bit, popular for deployment):
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(
        "TheBloke/Llama-2-7B-GPTQ",
        device_map="auto"
    )  # loads 4-bit quantized model

    # AWQ (4-bit, activation-aware):
    from awq import AutoAWQForCausalLM
    model = AutoAWQForCausalLM.from_quantized(
        "TheBloke/Llama-2-7B-AWQ"
    )

    # bitsandbytes (simple 4/8-bit loading):
    model = AutoModelForCausalLM.from_pretrained(
        "meta-llama/Llama-2-7b",
        load_in_4bit=True  # ← just this flag!
    )
    ''')
else:
    print('PyTorch not available for quantization demo')

---

## 8. Modern GPU Number Formats

### FP8 — Two Variants for Different Roles

| Format | Exponent | Mantissa | Range | Role |
|--------|----------|----------|-------|------|
| E4M3 | 4 | 3 | ±448 | Forward (weights, activations) |
| E5M2 | 5 | 2 | ±57344 | Backward (gradients) |

**Key insight**: Weights cluster tightly (need precision → E4M3), gradients vary wildly (need range → E5M2).

### TF32 — Hidden Default on A100/H100
- 19-bit: same range as FP32, same precision as FP16
- **Enabled by default** on A100+ for matmul
- Disable: `torch.backends.cuda.matmul.allow_tf32 = False`

In [ ]:
# The Complete Modern Format Landscape

print(f'{"Format":<12} {"Bits":>4} {"Exp":>4} {"Man":>4}  {"Speedup":>14}  {"Use Case"}')
print('='*72)
for name, bits, exp, man, speed, use in [
    ('FP32',     32, 8,  23, '1× baseline',  'Default training'),
    ('TF32',     19, 8,  10, '8× (A100+)',    'Auto matmul on A100'),
    ('BF16',     16, 8,   7, '16×',           'Training (preferred)'),
    ('FP16',     16, 5,  10, '16×',           'Inference, old AMP'),
    ('FP8 E4M3',  8, 4,   3, '32× (H100+)',  'Forward pass'),
    ('FP8 E5M2',  8, 5,   2, '32× (H100+)',  'Backward pass'),
    ('INT8',      8, '-', '-', '32×',         'PTQ inference'),
    ('INT4',      4, '-', '-', '64×',         'GPTQ/AWQ inference'),
]:
    print(f'  {name:<10} {bits:>4}  {str(exp):>4} {str(man):>4}  {speed:>14}  {use}')

# Show precision differences between E4M3 and E5M2
print('\n--- FP8 Precision Levels ---')
print('E4M3 (3 mantissa bits): values between 1.0 and 2.0:')
print(f'  {[1 + i/8 for i in range(8)]}  (8 levels, spacing 0.125)')
print('E5M2 (2 mantissa bits): values between 1.0 and 2.0:')
print(f'  {[1 + i/4 for i in range(4)]}  (4 levels, spacing 0.25)')

# Real FP8 code
print('\n--- FP8 Code (requires H100 + Transformer Engine) ---')
print('  import transformer_engine.pytorch as te')
print('  layer = te.Linear(4096, 4096)')
print('  with te.fp8_autocast(): output = layer(x)')

---

## 9. Numerical Algorithms for ML

### Stochastic Rounding
In low precision, deterministic rounding creates **bias** — small updates always round to zero.

**Stochastic rounding**: round probabilistically based on fractional part.
- Value 3.2 → round to 4 with prob 0.2, to 3 with prob 0.8
- E[round(3.2)] = 0.8×3 + 0.2×4 = 3.2 — **unbiased!**

### Kahan Summation
When summing millions of small numbers, floating-point errors compound.
Track rounding error and compensate each step.

In [ ]:
# Stochastic Rounding — Unbiased Low-Precision Updates

def stochastic_round(x):
    floor = np.floor(x)
    frac = x - floor
    return floor + (np.random.random(x.shape) < frac).astype(x.dtype)

np.random.seed(42)
values = np.full(10000, 3.2)
det = np.round(values)
stoch = stochastic_round(values)

print(f'Rounding 3.2 (10,000 times):')
print(f'  Deterministic: always → {int(det[0])}, mean = {det.mean():.4f}  ← BIASED')
print(f'  Stochastic:    mean = {stoch.mean():.4f}  ← UNBIASED ✓')

# Impact on gradient accumulation
print('\nGradient accumulation (100 updates of +0.15):')
w_det, w_stoch = 0.0, 0.0
for _ in range(100):
    w_det = np.round(w_det + 0.15)
    w_stoch = float(stochastic_round(np.array([w_stoch + 0.15])))
print(f'  Expected: {0.15*100:.1f}')
print(f'  Deterministic: {w_det:.1f}  ← updates LOST!')
print(f'  Stochastic:    {w_stoch:.1f}  ← updates preserved!')

In [ ]:
# Kahan Summation — Fixing Float32 Accumulation Error

def naive_sum(values):
    total = np.float32(0.0)
    for v in values:
        total += np.float32(v)
    return float(total)

def kahan_sum(values):
    """Compensated summation — tracks and corrects rounding error."""
    total = 0.0
    comp = 0.0  # compensation for lost low-order bits
    for v in values:
        y = float(v) - comp      # recover last error
        temp = total + y          # large + small (error here)
        comp = (temp - total) - y # capture what was lost
        total = temp
    return total

# Sum 1,000,000 small values in float32
n = 1_000_000
vals = np.full(n, 1e-5, dtype=np.float32)
true = 10.0

r_naive = naive_sum(vals)
r_kahan = kahan_sum(vals)
r_numpy = float(np.sum(vals))  # numpy uses pairwise summation

print(f'Summing {n:,} × 1e-5 (expected: 10.0):')
print(f'{"Method":<18} {"Result":>12} {"Error":>12}')
print('-'*44)
print(f'{"Naive float32":<18} {r_naive:>12.6f} {abs(r_naive-true):>12.6f}')
print(f'{"Kahan summation":<18} {r_kahan:>12.6f} {abs(r_kahan-true):>12.6f}')
print(f'{"NumPy (pairwise)":<18} {r_numpy:>12.6f} {abs(r_numpy-true):>12.6f}')
print(f'\n→ Use Kahan sum or float64 for gradient accumulation')
print('→ PyTorch uses float32 master weights even in mixed-precision training')

---

## Summary: Key Takeaways

| Concept | Why It Matters | Action |
|---------|----------------|--------|
| **IEEE 754 layout** | Debug NaN, precision bugs | Use `struct.pack` to inspect bits |
| **Machine epsilon** | Minimum useful learning rate | Check `np.finfo()` |
| **Softmax overflow** | NaN in logits | Always subtract max |
| **Log-space** | Underflow prevention | Sum of logs, not product of probs |
| **Mixed precision** | 2-3× training speedup | `torch.cuda.amp.autocast()` |
| **BFloat16 > Float16** | Wider range = fewer NaN | Use BF16 on Ampere+ GPUs |
| **Per-channel quant** | Production accuracy | Use over per-tensor |
| **GPTQ/AWQ** | 4-bit LLM deployment | `load_in_4bit=True` |
| **FP8 E4M3/E5M2** | 4× speedup on H100 | Transformer Engine |
| **Stochastic rounding** | Unbiased low-precision | Critical for FP8 training |
| **Kahan summation** | Accurate gradient accumulation | Compensate rounding errors |
| **RoPE (complex)** | Position encoding in LLMs | Complex rotation of Q/K vectors |

### Best Practices

1. **Never compare floats with `==`** → Use `np.isclose()` or `torch.allclose()`
2. **Large logits?** → Stable softmax (subtract max)
3. **Product of probabilities?** → Sum of log-probs
4. **Training large models?** → Mixed precision with BFloat16
5. **Deploying LLMs?** → GPTQ/AWQ 4-bit quantization
6. **On H100?** → FP8 via NVIDIA Transformer Engine
7. **Accumulating gradients?** → Float32 master weights or Kahan summation

---

## Practice Questions

1. **Why does `0.1 + 0.2 != 0.3`?** (IEEE 754 binary representation)
2. **What happens with `np.exp(1000)`?** (Overflow → inf)
3. **Why subtract max in softmax?** (Prevent exp overflow)
4. **Float32 → Int8: how much memory saved?** (4× for a 7B model: 28GB → 7GB)
5. **BFloat16 vs Float16: which for training?** (BFloat16 — same range as FP32)
6. **E4M3 vs E5M2 difference?** (Precision vs range for forward vs backward)
7. **Per-channel vs per-tensor quantization?** (Different scales per channel → less error)
8. **What does stochastic rounding solve?** (Bias in low-precision gradient updates)
9. **How does RoPE use complex numbers?** (Rotation in complex plane for position encoding)

---

### Next Steps

- **exercises.ipynb**: Practice problems with solutions
- **notes.md**: Detailed theory and mathematical background
- Continue to: **02-Sets-and-Logic**